# Bronze, reference build

> This is the **finished shape**, in the order you would actually build it. The
> learning version with the deliberate failures is [`02_bronze.ipynb`](02_bronze.ipynb).
> Read that one to understand *why*. Read this one to remember *how*.

**Iceberg is not a step here.** It is the destination, declared in part 3 and used by
everything after it. Bolting it on at the end was a teaching device, not a design.

## The order of thinking

```
   0  FRAME       what must be true when this is finished
   1  CONTRACT    what shape of input will I accept?          ── decide BEFORE writing
   2  ROW SHAPE   what does one row become?
   3  DESTINATION declare the table: schema + partition spec  ── decide BEFORE writing
   4  INGEST      read → check → enrich → commit              ── one function
   5  RUN         run it twice
   6  VERIFY      does the data say what I think it says?
   7  OBSERVE     snapshots, files, manifests
   8  RECOVER     time travel and rollback
   9  MAINTAIN    retention and file sizing
  10  ON DISK     what all of that actually looks like
```

Notice that **three of the first four steps are decisions, not code.** That is the
actual lesson of this layer.

## 0. Frame

Four properties. Everything below exists to make one of them true.

| Property | Means | How this notebook proves it |
|---|---|---|
| **Faithful** | Same rows in as out, good and bad alike | Part 6, row count equals source |
| **Idempotent** | Running twice equals running once | Part 5, same count both runs |
| **Observable** | You can tell what ran, when, over what | Part 7, snapshots and batch ids |
| **Atomic** | The whole write lands or none of it does | Part 3, Iceberg commits |

The decisions, made once and recorded in [`../docs/bronze.md`](../docs/bronze.md) §9:

| Decision | Chosen | Because |
|---|---|---|
| Idempotency | `overwritePartitions()` | Replaces only the months present. Row-level MERGE needs a key this data lacks |
| Partitioning | `year` + `month` | Matches the reprocessing unit: one source file is one month |
| Partition values from | The **filename** | The March file contains trips dated 2009. Judging that is silver's job |
| Schema drift | Accept additive, fail on missing or retyped | Additive cannot corrupt. The others corrupt silently |
| Cleaning | None at all | Dropping a row here destroys evidence |
| Format | Iceberg over Parquet | Atomic commits, snapshots, rollback |

In [ ]:
import sys, json
from pathlib import Path
from datetime import datetime, timezone

sys.path.insert(0, str(Path.cwd().parent))

from pyspark.sql import functions as F
from pyspark.sql.functions import col
from src.session import get_spark
from src import config

TABLE = "local.bronze.yellow"

spark = get_spark("bronze-reference", iceberg=True)
sc = spark.sparkContext
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.bronze")

print("months available:", config.available_months())

---

## 1. Contract

**Decided before any data is written.** The expected schema lives in code, not in the
data, because a check that compares a file to itself always passes.

| Change at the source | Policy | Why |
|---|---|---|
| New column | **Accept**, log it | Additive. Cannot corrupt what exists |
| Column missing | **Fail** | Becomes NULL downstream, and NULL is a real value in this domain |
| Type changed | **Fail** | Spark coerces silently. `"12"` sorts differently from `12` |
| Column reordered | **Ignore** | Parquet is name-addressed. ⚠️ Only safe because nothing here reads by position |

In [ ]:
EXPECTED = {
    "VendorID": "int",
    "tpep_pickup_datetime": "timestamp_ntz",
    "tpep_dropoff_datetime": "timestamp_ntz",
    "passenger_count": "bigint",
    "trip_distance": "double",
    "RatecodeID": "bigint",
    "store_and_fwd_flag": "string",
    "PULocationID": "int",
    "DOLocationID": "int",
    "payment_type": "bigint",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "Airport_fee": "double",
}


class SchemaContractError(Exception):
    '''Raised when incoming data breaks the agreed shape.'''


def check_schema(df, expected=EXPECTED, source="<unknown>"):
    '''Fail loudly if an incoming file does not match the contract.'''
    actual = {f.name: f.dataType.simpleString() for f in df.schema.fields}

    missing = sorted(set(expected) - set(actual))
    added   = sorted(set(actual) - set(expected))
    retyped = sorted(c for c in set(expected) & set(actual) if expected[c] != actual[c])

    if added:
        print(f"  NOTE  {source}: new column(s) {added}. Accepted, not yet used downstream.")

    problems = []
    if missing:
        problems.append(f"missing column(s): {missing}")
    if retyped:
        problems += [f"{c}: expected {expected[c]}, got {actual[c]}" for c in retyped]

    if problems:
        raise SchemaContractError(f"{source} breaks the contract -> " + "; ".join(problems))

In [ ]:
def expect(label, fn, should_raise):
    try:
        fn()
        print(f"{'FAIL' if should_raise else 'ok  '}  {label}")
    except SchemaContractError as e:
        print(f"{'ok  ' if should_raise else 'FAIL'}  {label}  ->  {e}")

probe = spark.read.parquet(str(config.raw_month_path(*config.available_months()[0])))

expect("unchanged      ", lambda: check_schema(probe), False)
expect("column added   ", lambda: check_schema(probe.withColumn("surge_fee", F.lit(1.0))), False)
expect("column missing ", lambda: check_schema(probe.drop("passenger_count")), True)
expect("column retyped ", lambda: check_schema(probe.withColumn("trip_distance", col("trip_distance").cast("string"))), True)

---

## 2. Row shape

Five columns added. **Three say where the row came from, two say where it goes.**

| Column | Answers | Note |
|---|---|---|
| `_source_file` | Which file produced this row? | The handle for a republished month |
| `_ingested_at` | When did we load it? | Constant within one query, so it varies **per month**, not per row |
| `_batch_id` | Which run produced this? | Constant across the whole run. **Not** the Spark job id, which dies with the session |
| `year`, `month` | Where on disk | From the **filename**, never from `tpep_pickup_datetime` |

Nothing is renamed, retyped, dropped or filtered. That is the whole rule of this layer.

In [ ]:
def enrich(df, year: int, month: int, batch_id: str):
    '''Add provenance and partition columns. Reshapes nothing.'''
    return (df
        .withColumn(config.COL_SOURCE_FILE, F.input_file_name())
        .withColumn(config.COL_INGESTED_AT, F.current_timestamp())
        .withColumn(config.COL_BATCH_ID,    F.lit(batch_id))
        .withColumn("year",  F.lit(year))
        .withColumn("month", F.lit(month)))

---

## 3. Destination

**The table is declared once, before any ingest runs.** Schema and partition spec are
the two things you own; Iceberg maintains everything else, forever, without being asked.

Creating it with `.limit(0)` registers the shape without landing data, so **snapshot 1
is an empty table.** That matters: the history then covers the entire life of the table
rather than starting from whenever you happened to bolt Iceberg on.

In [ ]:
y0, m0 = config.available_months()[0]

(enrich(spark.read.parquet(str(config.raw_month_path(y0, m0))), y0, m0, "init")
    .limit(0)
    .writeTo(TABLE)
    .partitionedBy(col("year"), col("month"))
    .createOrReplace())

print(f"{TABLE} created, {spark.table(TABLE).count()} rows, "
      f"{len(spark.table(TABLE).columns)} columns")
spark.sql(f"SELECT * FROM {TABLE}.snapshots").select("snapshot_id", "operation").show()

---

## 4. Ingest

**Four steps, in this order, and the order is the point.**

```
   read  ─►  CHECK  ─►  enrich  ─►  commit
             ^^^^^
             before anything is written. A contract breach must cost nothing.
```

`overwritePartitions()` replaces exactly the partitions present in the incoming
DataFrame and leaves every other one alone, **in a single atomic commit**. That is
`partitionOverwriteMode=dynamic`, plus atomicity, plus a snapshot.

In [ ]:
def land_month(year: int, month: int, batch_id: str) -> None:
    src_path = config.raw_month_path(year, month)

    raw = spark.read.parquet(str(src_path))          # 1. read
    check_schema(raw, source=src_path.name)          # 2. check, before anything is written
    enriched = enrich(raw, year, month, batch_id)    # 3. enrich

    sc.setJobDescription(f"bronze | {year}-{month:02d} | batch {batch_id}")
    enriched.writeTo(TABLE).overwritePartitions()    # 4. commit, atomically


def run_all() -> str:
    batch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    for y, m in config.available_months():
        land_month(y, m, batch_id)
    sc.setJobDescription(None)
    return batch_id

---

## 5. Run it twice

The only test of idempotency that counts.

In [ ]:
b1 = run_all()
n1 = spark.table(TABLE).count()
print(f"run 1   batch={b1}   rows={n1:,}")

b2 = run_all()
n2 = spark.table(TABLE).count()
print(f"run 2   batch={b2}   rows={n2:,}")

print("\nIDEMPOTENT" if n1 == n2 else "\nNOT IDEMPOTENT")

---

## 6. Verify

Three questions the data itself has to answer.

In [ ]:
# 1. Faithful: does every source row survive?
src_total = sum(
    spark.read.parquet(str(config.raw_month_path(y, m))).count()
    for y, m in config.available_months()
)
tbl_total = spark.table(TABLE).count()
print(f"source {src_total:,}   bronze {tbl_total:,}   {'MATCH' if src_total == tbl_total else 'MISMATCH'}")

# 2. Observable: one batch, one timestamp per month
print()
(spark.table(TABLE)
    .groupBy("year", "month", config.COL_BATCH_ID)
    .agg(F.count("*").alias("rows"),
         F.countDistinct(config.COL_INGESTED_AT).alias("distinct_ingested_at"))
    .orderBy("year", "month")
    .show(truncate=False))

# 3. Traceable: every row knows its file
print()
(spark.table(TABLE)
    .groupBy(F.element_at(F.split(col(config.COL_SOURCE_FILE), "/"), -1).alias("source_file"))
    .count().orderBy("source_file").show(truncate=False))

Read the second table carefully. **One `_batch_id`, but a different `_ingested_at` per
month.** `current_timestamp()` is fixed within a single query and each month is its own
query, so the batch id is the only thing that ties the three writes together as one run.
That is the entire reason the column exists.

---

## 7. Observe

The metadata tables. **This is what replaces listing directories.**

In [ ]:
print("=== SNAPSHOTS: every commit, in order ===")
spark.sql(f"""
  SELECT snapshot_id, parent_id, operation,
         summary['added-records']    AS added,
         summary['deleted-records']  AS deleted,
         summary['added-data-files'] AS files
  FROM {TABLE}.snapshots ORDER BY committed_at
""").show(truncate=False)

In [ ]:
print("=== FILES: a manifest, rendered as a table ===")
spark.sql(f"""
  SELECT partition, record_count, file_size_in_bytes,
         substring_index(file_path, '/', -1) AS file
  FROM {TABLE}.files ORDER BY partition
""").show(truncate=False)

print("=== MANIFESTS ===")
spark.sql(f"""
  SELECT substring_index(path, '/', -1) AS manifest, length,
         added_data_files_count AS added, existing_data_files_count AS existing,
         deleted_data_files_count AS deleted
  FROM {TABLE}.manifests
""").show(truncate=False)

The `deleted` count on the later snapshots is the proof that `overwritePartitions()`
**replaced** rather than appended. An append would show added files and zero deleted.

---

## 8. Recover

What you would otherwise do with `rm -rf` and a re-run.

In [ ]:
snaps = [r[0] for r in spark.sql(
    f"SELECT snapshot_id FROM {TABLE}.snapshots ORDER BY committed_at").collect()]

print("time travel, without changing anything:")
for s in snaps:
    n = spark.sql(f"SELECT count(*) AS c FROM {TABLE} VERSION AS OF {s}").first().c
    print(f"   snapshot {s}  ->  {n:,} rows")

In [ ]:
# Roll the live table back. This is the replacement for deleting directories.
target = snaps[0]
spark.sql(f"CALL local.system.rollback_to_snapshot('bronze.yellow', {target})")
print(f"rolled back to {target}  ->  {spark.table(TABLE).count():,} rows")

# A rollback is itself just another commit, so going forward is symmetric.
spark.sql(f"CALL local.system.set_current_snapshot('bronze.yellow', {snaps[-1]})")
print(f"forward to     {snaps[-1]}  ->  {spark.table(TABLE).count():,} rows")

---

## 9. Maintain

Nothing here happens automatically. **A table nobody maintains keeps every version
forever and the storage bill grows quietly.**

| Procedure | Removes | Consequence |
|---|---|---|
| `expire_snapshots` | Old snapshots, and data files no surviving snapshot references | Time travel stops working before that point |
| `remove_orphan_files` | Files nothing references, usually from failed commits | ⚠️ Use a conservative window. It can delete files a live writer is creating |
| `rewrite_data_files` | Nothing. Compacts small files into larger ones | Faster scans |
| `rewrite_manifests` | Nothing. Reorganises metadata | Faster planning after many small commits |

The real decision is a **retention policy**: how far back must you be able to travel,
and what is that worth in storage? Seven to thirty days is common.

In [ ]:
before = len(snaps)
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

# Retention policy: keep the 2 most recent snapshots. `older_than` is required, so
# pass "now" and let retain_last do the work.
spark.sql(f"""
  CALL local.system.expire_snapshots(
    table       => 'bronze.yellow',
    older_than  => TIMESTAMP '{now}',
    retain_last => 2)
""").show(truncate=False)

after = spark.sql(f"SELECT count(*) AS c FROM {TABLE}.snapshots").first().c
print(f"snapshots: {before} -> {after}")
print(f"rows still {spark.table(TABLE).count():,}, because the CURRENT snapshot is untouched")

---

## 10. What it looks like on disk

In [ ]:
meta = config.ROOT / "warehouse" / "bronze" / "yellow" / "metadata"
data = config.ROOT / "warehouse" / "bronze" / "yellow" / "data"

print("metadata/")
for f in sorted(meta.iterdir()):
    kind = ("MANIFEST LIST" if f.name.startswith("snap-")
            else "TABLE STATE" if f.suffix == ".json"
            else "POINTER" if f.name == "version-hint.text"
            else "MANIFEST" if f.suffix == ".avro" else "")
    print(f"  {f.stat().st_size:>9,}  {f.name:<52} {kind}")

print("\ndata/")
for f in sorted(data.rglob("*.parquet")):
    print(f"  {f.stat().st_size:>9,}  {f.relative_to(data)}")

In [ ]:
print("--- version-hint.text: THE CATALOG POINTER, made visible ---")
print(repr((meta / "version-hint.text").read_text()))

latest = max(meta.glob("v*.metadata.json"), key=lambda p: int(p.name[1:].split(".")[0]))
d = json.loads(latest.read_text())

print(f"\n--- {latest.name} ---")
print("format-version      :", d["format-version"])
print("current-snapshot-id :", d["current-snapshot-id"])
print("schemas             :", len(d["schemas"]))
print("partition spec      :", [(f["name"], f["transform"]) for f in d["partition-specs"][0]["fields"]])
print("\nsnapshots still referenced:")
for s in d["snapshots"]:
    op = s.get("summary", {}).get("operation", "?")   # NOTE: operation lives inside summary
    print(f"   {s['snapshot-id']}  {op:<9}  ->  {s['manifest-list'].split('/')[-1]}")

### Two things that will surprise you in that listing

**One data file per month, not two.** Plain Parquet gave you `part-00000` and
`part-00001` per month. Iceberg gives you one. Nothing in this notebook asked for that:
Iceberg's default `write.distribution-mode` for a partitioned table is `hash`, so it
shuffles by partition key before writing and each partition ends up written by a single
task. **The format solved the file-count problem you had to solve by hand.**

**`month=3` still has two files, of identical size.** `expire_snapshots` deleted two
data files but not those, because a snapshot you chose to retain still points at one of
them. That is time travel's storage cost, made visible: **you cannot go back to a
version whose files you deleted, so the files survive exactly as long as the snapshot
does.**

### The one thing to take away

**`version-hint.text` contains a single number.** That is the entire catalog pointer for
a hadoop catalog, and committing means writing a new number into that file.

Which is exactly why it cannot be trusted with concurrent writers: *"change 7 to 8, but
only if it is still 7"* is not something a filesystem can promise. A database can do it
in a transaction. A REST catalog can do it in memory. A file cannot.

That single line is the whole argument for why production Iceberg needs a real catalog.

---

## Done when

- [ ] The schema test prints `ok` on all four cases
- [ ] Two runs give an identical row count
- [ ] Bronze row count equals the source row count exactly
- [ ] Every row traces to a source file and a batch id
- [ ] You have rolled the table back and forward and watched the count move
- [ ] You have read `version-hint.text` and a real `metadata.json`

## Then lift it into `src/`

| Notebook part | Module |
|---|---|
| 1 | `src/bronze/schema.py` — `EXPECTED`, `check_schema` |
| 2, 4 | `src/bronze/ingest.py` — `enrich`, `land_month`, `main` |
| 3 | `src/bronze/ddl.py`, or a one-off migration script |
| 9 | `src/maintenance.py`, called weekly by Airflow, not per run |

Then `python -m src.bronze.ingest` twice, and the row count does not move.